# **Raquel Rocha**
### **Projeto Integrador — Modelagem e Inserção no Banco de Dados**

---

# **0.0 Imports**

In [26]:
import sqlite3
import pandas as pd
import os
from datetime import datetime

## **0.1 Loading Data**

Carregamento dos CSVs gerados na etapa de coleta.

In [27]:
df_deputados     = pd.read_csv('deputados.csv', low_memory=False)
df_tipos_despesa = pd.read_csv('tipos_despesa.csv', low_memory=False)
df_despesas      = pd.read_csv('despesas.csv', low_memory=False)

print('Number of Rows deputados     : {}'.format(df_deputados.shape[0]))
print('Number of Rows tipos_despesa : {}'.format(df_tipos_despesa.shape[0]))
print('Number of Rows despesas      : {}'.format(df_despesas.shape[0]))

Number of Rows deputados     : 641
Number of Rows tipos_despesa : 23
Number of Rows despesas      : 668911


---
# **1.0 Tratamento dos Dados**

## **1.1 Tabela: tipos_despesa**

In [28]:
df_td = df_tipos_despesa.copy()

# mantém apenas as colunas relevantes
colunas_td = [c for c in ['cod', 'nome'] if c in df_td.columns]
df_td = df_td[colunas_td].copy()

# remove duplicatas pela PK (nome)
df_td = df_td.drop_duplicates(subset=['nome']).reset_index(drop=True)

# ── CORREÇÃO: garante que todos os tipos presentes em despesas existam em tipos_despesa
# (ex: "PASSAGEM AÉREA - SIGEPA" existe nas despesas mas não no CSV de referência)
tipos_em_despesas = set(df_despesas['tipo_despesa'].dropna().unique())
tipos_em_td       = set(df_td['nome'].unique())
tipos_faltantes   = tipos_em_despesas - tipos_em_td

if tipos_faltantes:
    print('Tipos encontrados em despesas mas ausentes em tipos_despesa (serão adicionados):')
    extras = []
    for nome in sorted(tipos_faltantes):
        print('  +', nome)
        extras.append({'cod': None, 'nome': nome})
    df_td = pd.concat([df_td, pd.DataFrame(extras)], ignore_index=True)

print('Registros após tratamento: {}'.format(len(df_td)))
print('Nulos:')
print(df_td.isnull().sum())

Tipos encontrados em despesas mas ausentes em tipos_despesa (serão adicionados):
  + PASSAGEM AÉREA - SIGEPA
Registros após tratamento: 24
Nulos:
cod     1
nome    0
dtype: int64


In [29]:
df_td.head()

,cod,nome
0,1,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...
1,2,"LOCOMOÇÃO, ALIMENTAÇÃO E HOSPEDAGEM"
2,3,COMBUSTÍVEIS E LUBRIFICANTES.
3,4,"CONSULTORIAS, PESQUISAS E TRABALHOS TÉCNICOS."
4,5,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.


## **1.2 Tabela: deputados**

In [30]:
df_dep = df_deputados.copy()

# garante que id é string (identificador, não número)
df_dep['id_deputado'] = df_dep['id_deputado'].astype(str)

# preenche nulos
df_dep['partido'] = df_dep['partido'].fillna('SEM PARTIDO')
df_dep['uf']      = df_dep['uf'].fillna('NAO INFORMADO')
df_dep['nome']    = df_dep['nome'].fillna('NAO INFORMADO')

# mantém apenas as colunas do modelo
colunas_dep = [c for c in ['id_deputado', 'nome', 'partido', 'uf', 'uri'] if c in df_dep.columns]
df_dep = df_dep[colunas_dep].copy()

# remove duplicatas pela chave primária
df_dep = df_dep.drop_duplicates(subset=['id_deputado']).reset_index(drop=True)

print('Registros após tratamento: {}'.format(len(df_dep)))
print('Nulos:')
print(df_dep.isnull().sum())

Registros após tratamento: 641
Nulos:
id_deputado    0
nome           0
partido        0
uf             0
uri            0
dtype: int64


In [31]:
df_dep.head()

,id_deputado,nome,partido,uf,uri
0,220593,Abilio Brunini,PL,MT,https://dadosabertos.camara.leg.br/api/v2/depu...
1,204379,Acácio Favacho,MDB,AP,https://dadosabertos.camara.leg.br/api/v2/depu...
2,220714,Adail Filho,REPUBLICANOS,AM,https://dadosabertos.camara.leg.br/api/v2/depu...
3,221328,Adilson Barroso,PL,SP,https://dadosabertos.camara.leg.br/api/v2/depu...
4,204560,Adolfo Viana,PSDB,BA,https://dadosabertos.camara.leg.br/api/v2/depu...


## **1.3 Tabela: despesas**

In [32]:
df_desp = df_despesas.copy()

# tipos corretos
df_desp['id_deputado']          = df_desp['id_deputado'].astype(str)
df_desp['cod_documento']        = df_desp['cod_documento'].astype(str)
df_desp['cnpj_cpf_fornecedor']  = df_desp['cnpj_cpf_fornecedor'].fillna('NAO INFORMADO').astype(str)
df_desp['nome_fornecedor']      = df_desp['nome_fornecedor'].fillna('NAO INFORMADO').astype(str)
df_desp['url_documento']        = df_desp['url_documento'].fillna('NAO INFORMADO').astype(str)
df_desp['num_documento']        = df_desp['num_documento'].fillna('NAO INFORMADO').astype(str)

# data como string ISO para o SQLite (YYYY-MM-DD)
df_desp['data_documento'] = pd.to_datetime(df_desp['data_documento'], errors='coerce')
df_desp['data_documento'] = df_desp['data_documento'].dt.strftime('%Y-%m-%d')

# valores financeiros como float
for col in ['valor_documento', 'valor_liquido', 'valor_glosa']:
    df_desp[col] = pd.to_numeric(df_desp[col], errors='coerce').fillna(0.0)

# ano e mês como inteiro
df_desp['ano'] = pd.to_numeric(df_desp['ano'], errors='coerce').fillna(0).astype(int)
df_desp['mes'] = pd.to_numeric(df_desp['mes'], errors='coerce').fillna(0).astype(int)

# mantém apenas as colunas do modelo
colunas_desp = [c for c in [
    'cod_documento', 'id_deputado', 'tipo_despesa',
    'ano', 'mes', 'data_documento',
    'valor_documento', 'valor_liquido', 'valor_glosa',
    'nome_fornecedor', 'cnpj_cpf_fornecedor',
    'num_documento', 'url_documento'
] if c in df_desp.columns]
df_desp = df_desp[colunas_desp].copy()

# remove duplicatas pela chave primária composta
df_desp = df_desp.drop_duplicates(subset=['cod_documento', 'id_deputado']).reset_index(drop=True)

# remove linhas sem valor_liquido
df_desp = df_desp.dropna(subset=['valor_liquido'])

print('Registros após tratamento: {}'.format(len(df_desp)))
print('Nulos restantes:')
print(df_desp.isnull().sum())

Registros após tratamento: 668911
Nulos restantes:
cod_documento          0
id_deputado            0
tipo_despesa           0
ano                    0
mes                    0
data_documento         0
valor_documento        0
valor_liquido          0
valor_glosa            0
nome_fornecedor        0
cnpj_cpf_fornecedor    0
num_documento          0
url_documento          0
dtype: int64


In [33]:
df_desp.dtypes

,0
cod_documento,object
id_deputado,object
tipo_despesa,object
ano,int64
mes,int64
data_documento,object
valor_documento,float64
valor_liquido,float64
valor_glosa,float64
nome_fornecedor,object


In [34]:
df_desp.sample(3)

,cod_documento,id_deputado,tipo_despesa,ano,mes,data_documento,valor_documento,valor_liquido,valor_glosa,nome_fornecedor,cnpj_cpf_fornecedor,num_documento,url_documento
186284,7898845,220553,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2025,3,2025-03-18,4469.87,4432.62,37.25,INOVACAO NEGOCIOS IMOBILIARIOS LTDA,1734687000151.0,140000000002395946,https://www.camara.leg.br/cota-parlamentar/doc...
485059,7743987,220703,COMBUSTÍVEIS E LUBRIFICANTES.,2024,5,2024-05-06,400.00,400.00,0.00,Posto de Combustiveis Garantia Ltda,72578438000243.0,20517,http://www.camara.leg.br/cota-parlamentar/nota...
361916,7754271,220559,FORNECIMENTO DE ALIMENTAÇÃO DO PARLAMENTAR,2024,6,2024-06-12,25.90,25.90,0.00,ARCOS DOURADOS COMERCIO DE ALIMENTOS SA,42591651010530.0,1114452,http://www.camara.leg.br/cota-parlamentar/nota...


---
# **2.0 Criação do Banco de Dados**

Usamos SQLite por ser simples, sem necessidade de servidor e compatível com o Colab.

O modelo segue exatamente o DER definido no projeto:

• `tipos_despesa` — tabela de referência (sem FK)

• `deputados` — tabela de parlamentares (sem FK)

• `despesas` — tabela principal com FK para `deputados` (id_deputado) e `tipos_despesa` (tipo_despesa)

## **2.1 Conexão**

In [35]:
DB_PATH = 'gastos_parlamentares.db'

# remove banco anterior se existir (evita conflito ao re-executar)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print('Banco anterior removido.')

conn   = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ativa suporte a chaves estrangeiras no SQLite
cursor.execute('PRAGMA foreign_keys = ON')
# melhora performance em inserções grandes
cursor.execute('PRAGMA journal_mode = WAL')

print('Conexão estabelecida: {}'.format(DB_PATH))

Conexão estabelecida: gastos_parlamentares.db


## **2.2 Criação das Tabelas**

In [36]:
# ── TABELA 1: tipos_despesa (referência)
# IMPORTANTE: 'nome' é a PRIMARY KEY porque a tabela despesas referencia pelo nome,
# não pelo cod. O SQLite exige que a coluna referenciada por FK seja a PRIMARY KEY.
cursor.execute('''
    CREATE TABLE IF NOT EXISTS tipos_despesa (
        cod  TEXT,
        nome TEXT PRIMARY KEY
    )
''')

# ── TABELA 2: deputados
cursor.execute('''
    CREATE TABLE IF NOT EXISTS deputados (
        id_deputado TEXT PRIMARY KEY,
        nome        TEXT NOT NULL,
        partido     TEXT,
        uf          TEXT,
        uri         TEXT
    )
''')

# ── TABELA 3: despesas (tabela principal)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS despesas (
        cod_documento       TEXT,
        id_deputado         TEXT NOT NULL,
        tipo_despesa        TEXT,
        ano                 INTEGER,
        mes                 INTEGER,
        data_documento      TEXT,
        valor_documento     REAL,
        valor_liquido       REAL,
        valor_glosa         REAL,
        nome_fornecedor     TEXT,
        cnpj_cpf_fornecedor TEXT,
        num_documento       TEXT,
        url_documento       TEXT,
        PRIMARY KEY (cod_documento, id_deputado),
        FOREIGN KEY (id_deputado)  REFERENCES deputados(id_deputado),
        FOREIGN KEY (tipo_despesa) REFERENCES tipos_despesa(nome)
    )
''')

conn.commit()
print('Tabelas criadas com sucesso!')

Tabelas criadas com sucesso!


---
# **3.0 Inserção dos Dados**

A ordem de inserção respeita as dependências do DER:

1. `tipos_despesa` (sem dependências)
2. `deputados` (sem dependências)
3. `despesas` (depende das duas anteriores)

## **3.1 Inserção: tipos_despesa**

In [37]:
print('Inserindo tipos_despesa...')

registros_inseridos = 0
registros_ignorados = 0

for _, row in df_td.iterrows():
    try:
        cursor.execute('''
            INSERT OR IGNORE INTO tipos_despesa (cod, nome)
            VALUES (?, ?)
        ''', (
            str(row['cod']) if pd.notna(row.get('cod')) else None,
            str(row['nome']) if pd.notna(row.get('nome')) else None,
        ))
        if cursor.rowcount > 0:
            registros_inseridos += 1
        else:
            registros_ignorados += 1
    except Exception as e:
        print('Erro na linha {}: {}'.format(_, e))

conn.commit()
print('Inseridos : {}'.format(registros_inseridos))
print('Ignorados : {} (duplicatas)'.format(registros_ignorados))

Inserindo tipos_despesa...
Inseridos : 24
Ignorados : 0 (duplicatas)


## **3.2 Inserção: deputados**

In [38]:
print('Inserindo deputados...')

registros_inseridos = 0
registros_ignorados = 0

for _, row in df_dep.iterrows():
    try:
        cursor.execute('''
            INSERT OR IGNORE INTO deputados (id_deputado, nome, partido, uf, uri)
            VALUES (?, ?, ?, ?, ?)
        ''', (
            str(row['id_deputado']),
            str(row['nome']),
            str(row.get('partido', 'SEM PARTIDO')),
            str(row.get('uf', 'NAO INFORMADO')),
            str(row['uri']) if 'uri' in row and pd.notna(row['uri']) else None,
        ))
        if cursor.rowcount > 0:
            registros_inseridos += 1
        else:
            registros_ignorados += 1
    except Exception as e:
        print('Erro na linha {}: {}'.format(_, e))

conn.commit()
print('Inseridos : {}'.format(registros_inseridos))
print('Ignorados : {} (duplicatas)'.format(registros_ignorados))

Inserindo deputados...
Inseridos : 641
Ignorados : 0 (duplicatas)


## **3.3 Inserção: despesas**

In [39]:
print('Inserindo despesas em lotes...')

TAMANHO_LOTE = 5000  # inserção em lotes de 5.000 linhas para evitar travamentos
total        = len(df_desp)
inseridos    = 0
ignorados    = 0
erros        = 0

for inicio in range(0, total, TAMANHO_LOTE):
    lote = df_desp.iloc[inicio:inicio + TAMANHO_LOTE]

    for _, row in lote.iterrows():
        try:
            cursor.execute('''
                INSERT OR IGNORE INTO despesas (
                    cod_documento, id_deputado, tipo_despesa,
                    ano, mes, data_documento,
                    valor_documento, valor_liquido, valor_glosa,
                    nome_fornecedor, cnpj_cpf_fornecedor,
                    num_documento, url_documento
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                str(row['cod_documento']),
                str(row['id_deputado']),
                str(row['tipo_despesa'])        if pd.notna(row.get('tipo_despesa'))        else None,
                int(row['ano'])                 if pd.notna(row.get('ano'))                 else None,
                int(row['mes'])                 if pd.notna(row.get('mes'))                 else None,
                str(row['data_documento'])      if pd.notna(row.get('data_documento'))      else None,
                float(row['valor_documento'])   if pd.notna(row.get('valor_documento'))     else None,
                float(row['valor_liquido'])     if pd.notna(row.get('valor_liquido'))       else None,
                float(row['valor_glosa'])       if pd.notna(row.get('valor_glosa'))         else None,
                str(row['nome_fornecedor'])     if pd.notna(row.get('nome_fornecedor'))     else None,
                str(row['cnpj_cpf_fornecedor']) if pd.notna(row.get('cnpj_cpf_fornecedor')) else None,
                str(row['num_documento'])       if pd.notna(row.get('num_documento'))       else None,
                str(row['url_documento'])       if pd.notna(row.get('url_documento'))       else None,
            ))
            if cursor.rowcount > 0:
                inseridos += 1
            else:
                ignorados += 1
        except Exception as e:
            erros += 1
            print(f'Erro na linha {_}: {e}')

    conn.commit()
    progresso = min(inicio + TAMANHO_LOTE, total)
    print('  [{}/{}] inseridos: {:,} | ignorados: {:,} | erros: {}'.format(
        progresso, total, inseridos, ignorados, erros))

print('\nInserção finalizada!')
print('  Inseridos : {:,}'.format(inseridos))
print('  Ignorados : {:,} (duplicatas)'.format(ignorados))
print('  Erros     : {}'.format(erros))

Inserindo despesas em lotes...
  [5000/668911] inseridos: 5,000 | ignorados: 0 | erros: 0
  [10000/668911] inseridos: 10,000 | ignorados: 0 | erros: 0
  [15000/668911] inseridos: 15,000 | ignorados: 0 | erros: 0
  [20000/668911] inseridos: 20,000 | ignorados: 0 | erros: 0
  [25000/668911] inseridos: 25,000 | ignorados: 0 | erros: 0
  [30000/668911] inseridos: 30,000 | ignorados: 0 | erros: 0
  [35000/668911] inseridos: 35,000 | ignorados: 0 | erros: 0
  [40000/668911] inseridos: 40,000 | ignorados: 0 | erros: 0
  [45000/668911] inseridos: 45,000 | ignorados: 0 | erros: 0
  [50000/668911] inseridos: 50,000 | ignorados: 0 | erros: 0
  [55000/668911] inseridos: 55,000 | ignorados: 0 | erros: 0
  [60000/668911] inseridos: 60,000 | ignorados: 0 | erros: 0
  [65000/668911] inseridos: 65,000 | ignorados: 0 | erros: 0
  [70000/668911] inseridos: 70,000 | ignorados: 0 | erros: 0
  [75000/668911] inseridos: 75,000 | ignorados: 0 | erros: 0
  [80000/668911] inseridos: 80,000 | ignorados: 0 | erro

---
# **4.0 Validação do Banco**

## **4.1 Contagem de Registros**

In [40]:
print('Validando contagem de registros:\n')

tabelas = ['tipos_despesa', 'deputados', 'despesas']

for tabela in tabelas:
    cursor.execute('SELECT COUNT(*) FROM {}'.format(tabela))
    total = cursor.fetchone()[0]
    print('  {:<20}: {:>10,} registros'.format(tabela, total))

Validando contagem de registros:

  tipos_despesa       :         24 registros
  deputados           :        641 registros
  despesas            :    668,911 registros


## **4.2 Amostra das Tabelas**

In [41]:
print('Amostra — tipos_despesa:')
display(pd.read_sql('SELECT * FROM tipos_despesa LIMIT 5', conn))

Amostra — tipos_despesa:


,cod,nome
0,1,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...
1,2,"LOCOMOÇÃO, ALIMENTAÇÃO E HOSPEDAGEM"
2,3,COMBUSTÍVEIS E LUBRIFICANTES.
3,4,"CONSULTORIAS, PESQUISAS E TRABALHOS TÉCNICOS."
4,5,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.


In [42]:
print('Amostra — deputados:')
display(pd.read_sql('SELECT * FROM deputados LIMIT 5', conn))

Amostra — deputados:


,id_deputado,nome,partido,uf,uri
0,220593,Abilio Brunini,PL,MT,https://dadosabertos.camara.leg.br/api/v2/depu...
1,204379,Acácio Favacho,MDB,AP,https://dadosabertos.camara.leg.br/api/v2/depu...
2,220714,Adail Filho,REPUBLICANOS,AM,https://dadosabertos.camara.leg.br/api/v2/depu...
3,221328,Adilson Barroso,PL,SP,https://dadosabertos.camara.leg.br/api/v2/depu...
4,204560,Adolfo Viana,PSDB,BA,https://dadosabertos.camara.leg.br/api/v2/depu...


In [43]:
print('Amostra — despesas:')
display(pd.read_sql('SELECT * FROM despesas LIMIT 5', conn))

Amostra — despesas:


,cod_documento,id_deputado,tipo_despesa,ano,mes,data_documento,valor_documento,valor_liquido,valor_glosa,nome_fornecedor,cnpj_cpf_fornecedor,num_documento,url_documento
0,7610066,220593,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2023,9,2023-09-19,52.98,52.98,0.0,3XIS COMERCIO VAREJISTA E ATACADISTA DE PAPELA...,42999796000188.0,35537,http://www.camara.leg.br/cota-parlamentar/nota...
1,7657491,220593,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2023,11,2023-11-27,67.30,67.30,0.0,AGUAS CUIABA S.A,14995581000153.0,01,https://www.camara.leg.br/cota-parlamentar/doc...
2,7552621,220593,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2023,5,2023-05-05,43.20,43.20,0.0,AGUAS CUIABA S.A,14995581000153.0,11533052023001,https://www.camara.leg.br/cota-parlamentar/doc...
3,7587236,220593,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2023,6,2023-06-05,43.20,43.20,0.0,AGUAS CUIABA S.A,14995581000153.0,11533062023001,https://www.camara.leg.br/cota-parlamentar/doc...
4,7587239,220593,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2023,7,2023-07-05,43.20,43.20,0.0,AGUAS CUIABA S.A,14995581000153.0,11533072023001,https://www.camara.leg.br/cota-parlamentar/doc...


## **4.3 Verificação das Chaves Estrangeiras**

In [44]:
# verifica se todas as despesas têm id_deputado válido na tabela deputados
cursor.execute('''
    SELECT COUNT(*) FROM despesas d
    LEFT JOIN deputados dep ON d.id_deputado = dep.id_deputado
    WHERE dep.id_deputado IS NULL
''')
orfaos_dep = cursor.fetchone()[0]
print('Despesas sem deputado correspondente: {}'.format(orfaos_dep))

# verifica se todas as despesas têm tipo_despesa válido na tabela tipos_despesa
cursor.execute('''
    SELECT COUNT(*) FROM despesas d
    LEFT JOIN tipos_despesa td ON d.tipo_despesa = td.nome
    WHERE td.nome IS NULL
''')
orfaos_td = cursor.fetchone()[0]
print('Despesas sem tipo_despesa correspondente: {}'.format(orfaos_td))

Despesas sem deputado correspondente: 0
Despesas sem tipo_despesa correspondente: 0


## **4.4 Consultas de Verificação**

In [45]:
# top 5 categorias por valor total
query = '''
    SELECT
        tipo_despesa,
        COUNT(*)                     AS qtd_registros,
        ROUND(SUM(valor_liquido), 2) AS valor_total
    FROM despesas
    GROUP BY tipo_despesa
    ORDER BY valor_total DESC
    LIMIT 5
'''
print('Top 5 categorias por valor total:')
display(pd.read_sql(query, conn))

Top 5 categorias por valor total:


,tipo_despesa,qtd_registros,valor_total
0,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,55934,3.283753e+08
1,LOCAÇÃO OU FRETAMENTO DE VEÍCULOS AUTOMOTORES,23722,1.346449e+08
2,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,65460,1.049202e+08
3,PASSAGEM AÉREA - SIGEPA,134085,9.650881e+07
4,COMBUSTÍVEIS E LUBRIFICANTES.,234616,7.203610e+07


In [46]:
# top 5 deputados por total gasto
query = '''
    SELECT
        dep.nome,
        dep.partido,
        dep.uf,
        COUNT(d.cod_documento)         AS qtd_despesas,
        ROUND(SUM(d.valor_liquido), 2) AS total_gasto
    FROM despesas d
    JOIN deputados dep ON d.id_deputado = dep.id_deputado
    GROUP BY dep.id_deputado
    ORDER BY total_gasto DESC
    LIMIT 5
'''
print('Top 5 deputados por total gasto:')
display(pd.read_sql(query, conn))

Top 5 deputados por total gasto:


,nome,partido,uf,qtd_despesas,total_gasto
0,Pompeo de Mattos,PDT,RS,2646,2143862.25
1,Vinicius Gurgel,PL,AP,454,2030671.90
2,Albuquerque,REPUBLICANOS,RR,670,2019910.30
3,João Maia,PL,RN,1145,1952186.76
4,Gabriel Mota,REPUBLICANOS,RR,210,1941739.14


In [47]:
# total gasto por ano
query = '''
    SELECT
        ano,
        COUNT(*)                     AS qtd_registros,
        ROUND(SUM(valor_liquido), 2) AS valor_total
    FROM despesas
    GROUP BY ano
    ORDER BY ano
'''
print('Total gasto por ano:')
display(pd.read_sql(query, conn))

Total gasto por ano:


,ano,qtd_registros,valor_total
0,2023,199352,2.246338e+08
1,2024,202873,2.381366e+08
2,2025,204681,2.407202e+08
3,2026,62005,7.863007e+07


---
# **5.0 Índices**

Criamos índices nas colunas mais consultadas para melhorar o desempenho das queries.

In [48]:
print('Criando índices...')

indices = [
    ('idx_despesas_id_deputado',        'despesas', 'id_deputado'),
    ('idx_despesas_tipo_despesa',        'despesas', 'tipo_despesa'),
    ('idx_despesas_ano_mes',             'despesas', 'ano, mes'),
    ('idx_despesas_valor_liquido',       'despesas', 'valor_liquido'),
    ('idx_despesas_nome_fornecedor',     'despesas', 'nome_fornecedor'),
    ('idx_despesas_cnpj_cpf_fornecedor', 'despesas', 'cnpj_cpf_fornecedor'),
]

for nome_idx, tabela, coluna in indices:
    cursor.execute('CREATE INDEX IF NOT EXISTS {} ON {} ({})'.format(
        nome_idx, tabela, coluna
    ))
    print('  ✓ {}'.format(nome_idx))

conn.commit()
print('\nÍndices criados!')

Criando índices...
  ✓ idx_despesas_id_deputado
  ✓ idx_despesas_tipo_despesa
  ✓ idx_despesas_ano_mes
  ✓ idx_despesas_valor_liquido
  ✓ idx_despesas_nome_fornecedor
  ✓ idx_despesas_cnpj_cpf_fornecedor

Índices criados!


---
# **6.0 Encerramento**

In [49]:
conn.close()

tamanho_mb = os.path.getsize(DB_PATH) / (1024 * 1024)

print('Banco de dados salvo: {}'.format(DB_PATH))
print('Tamanho             : {:.2f} MB'.format(tamanho_mb))
print('Data/hora           : {}'.format(datetime.now().strftime('%d/%m/%Y %H:%M:%S')))

Banco de dados salvo: gastos_parlamentares.db
Tamanho             : 250.77 MB
Data/hora           : 24/05/2026 22:53:54


## **6.1 Download do banco**

In [50]:
from google.colab import files
files.download(DB_PATH)
print('Download iniciado!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download iniciado!
